# 11. Anomaly Detection in Time Series

## Background

Time series anomaly detection leverages forecasting in a powerful way: train a model on 'normal' behavior, then flag observations where actuals deviate significantly from forecasts. The residuals — |actual - forecast| — are the anomaly score. Large residuals indicate unexpected behavior: equipment failures, fraud, cyberattacks, or data quality issues.

This **forecast-residual** approach has major advantages over classical methods like isolation forests or LOF (Local Outlier Factor): it captures temporal context, handles seasonality naturally, and produces interpretable anomaly scores (the residual is directly interpretable as 'how far from expected behavior').

Modern approaches include:
- **Anomaly Transformer** (Xu et al., 2021): attention association discrepancy
- **TranAD** (Tuli et al., 2022): adversarial training for multivariate anomaly detection
- **FITS** (Zhou et al., 2023): low-parameter frequency-domain model

## What You'll Learn

- Forecast-residual anomaly detection pipeline
- Dynamic thresholding: adaptive anomaly scores
- Point vs contextual vs collective anomalies
- Evaluation: precision, recall, and the F1-score for anomaly detection

## Why This Matters

Anomaly detection in time series is operationally critical: manufacturing downtime prevention, financial fraud detection, network intrusion, and cloud infrastructure monitoring all rely on detecting deviations from normal temporal patterns. Forecast-based detection with calibrated thresholds reduces alert fatigue compared to naive statistical thresholding.


## Generating Data with Injected Anomalies

In [ ]:
import numpy as np
import pandas as pd
from typing import Tuple, List

np.random.seed(42)

def generate_ts_with_anomalies(n: int = 500) -> Tuple[np.ndarray, np.ndarray]:
    '''Generate a time series with injected point and contextual anomalies.
    Returns (series, labels) where labels[i]=1 indicates anomaly.
    '''
    t = np.arange(n)
    series = 100 + 0.05*t + 10*np.sin(2*np.pi*t/52) + np.random.normal(0, 2, n)
    labels = np.zeros(n, dtype=int)

    # Inject point anomalies (sudden spikes)
    point_anomaly_idx = [50, 150, 280, 420]
    for idx in point_anomaly_idx:
        series[idx] += np.random.choice([-1, 1]) * np.random.uniform(20, 35)
        labels[idx] = 1

    # Inject contextual anomalies (level shift over 5-10 points)
    contextual_start = [100, 350]
    for start in contextual_start:
        length = np.random.randint(5, 12)
        series[start:start+length] += np.random.uniform(15, 25)
        labels[start:start+length] = 1

    # Inject collective anomaly (unusual pattern)
    series[200:215] = series[200:215] + 20 * np.sin(np.linspace(0, 2*np.pi, 15))
    labels[200:215] = 1

    return series, labels

series, true_labels = generate_ts_with_anomalies(500)

print(f'Time series with anomalies:')
print(f'  Length: {len(series)}')
print(f'  Anomalous points: {true_labels.sum()} ({true_labels.mean()*100:.1f}%)')
print(f'  Normal range: [{series[true_labels==0].min():.1f}, {series[true_labels==0].max():.1f}]')
print(f'  Anomaly range: [{series[true_labels==1].min():.1f}, {series[true_labels==1].max():.1f}]')


## Forecast-Residual Anomaly Detector

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

class ForecastResidualDetector:
    def __init__(self, context_len: int = 52, n_sigma: float = 3.0,
                 dynamic_window: int = 50):
        self.context_len = context_len
        self.n_sigma = n_sigma
        self.dynamic_window = dynamic_window

    def fit_predict(self, series: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        '''Rolling one-step-ahead forecasting, flag anomalies by residual.'''
        n = len(series)
        anomaly_scores = np.zeros(n)
        predictions = np.full(n, np.nan)

        for t in range(self.context_len, n):
            train = series[:t]

            # Holt-Winters one-step forecast
            try:
                model = ExponentialSmoothing(train, trend='add', seasonal='add',
                                             seasonal_periods=52, damped_trend=True)
                fit = model.fit(optimized=True)
                pred = fit.forecast(1)[0]
            except Exception:
                pred = train[-1]  # fallback: last value

            predictions[t] = pred

            # Dynamic threshold: sigma from recent residuals
            if t >= self.context_len + self.dynamic_window:
                recent_residuals = np.abs(series[t-self.dynamic_window:t] - predictions[t-self.dynamic_window:t])
                recent_residuals = recent_residuals[~np.isnan(recent_residuals)]
                if len(recent_residuals) > 5:
                    threshold = np.mean(recent_residuals) + self.n_sigma * np.std(recent_residuals)
                else:
                    threshold = self.n_sigma * np.std(train)
            else:
                threshold = self.n_sigma * np.std(series[:t])

            residual = abs(series[t] - pred)
            anomaly_scores[t] = residual / (threshold + 1e-8)  # normalized score

        return anomaly_scores, predictions

detector = ForecastResidualDetector(context_len=52, n_sigma=3.0, dynamic_window=20)
print('Running forecast-residual anomaly detection (may take ~30s)...')
scores, preds = detector.fit_predict(series)

# Threshold: score > 1.0 => anomaly (residual > 3-sigma)
pred_labels = (scores > 1.0).astype(int)

# Evaluate (skip context window)
start = 52
from sklearn.metrics import precision_score, recall_score, f1_score
precision = precision_score(true_labels[start:], pred_labels[start:])
recall = recall_score(true_labels[start:], pred_labels[start:])
f1 = f1_score(true_labels[start:], pred_labels[start:])

print(f'\nAnomaly Detection Results:')
print(f'  Precision: {precision:.3f} ({int(precision*100)}% of flagged anomalies are real)')
print(f'  Recall:    {recall:.3f} ({int(recall*100)}% of real anomalies detected)')
print(f'  F1 score:  {f1:.3f}')
print(f'  Flagged:   {pred_labels[start:].sum()} points')
print(f'  True:      {true_labels[start:].sum()} points')
